# Learning without Forgetting (LwF): ImageNet → MIT Indoor Scenes

This notebook is a cleaned and more rigorous reproduction pipeline for the **ImageNet → MIT Indoor 67 Scenes** experiment.

It compares:

1. Fine-Tuning  
2. Feature Extraction  
3. Learning without Forgetting (LwF)  

Main improvements over the previous notebook:

- AlexNet is used consistently.
- The official Scenes training split is divided into **train** and **validation** subsets.
- The official Scenes test set is evaluated only after model selection.
- Training uses augmentation; validation and testing use center crops.
- Fine-Tuning and LwF start from the same warmed-up new-task head.
- LwF includes the warm-up stage described in the paper.
- Feature Extraction uses a two-layer task-specific head and disables dropout in the frozen feature extractor.
- The best checkpoint is selected using validation accuracy.
-The final experiment is executed once with a fixed random seed
(seed = 42) due to computational constraints.
- ImageNet accuracy is evaluated on the first **80 validation batches**, consistently with the CUB experiment.


# Step 0 — Environment Setup

Import the required libraries, define the experiment configuration, set random seeds, and select a GPU when one is available.

For a quick code check, set `QUICK_MODE = True`.  
For the final results that should be reported in the project, keep `QUICK_MODE = False`.


In [ ]:
import os
import gc
import copy
import random
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image, ImageFile
from IPython.display import display
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True

# -----------------------------
# Experiment configuration
# -----------------------------

QUICK_MODE = False

RUN_SEEDS = [42]

VAL_SPLIT_SEED = 123
VAL_FRACTION = 0.15

BATCH_SIZE_SCENES = 32
BATCH_SIZE_IMAGENET = 64

# ImageNet is evaluated on the first 80 validation batches,
# consistently with the CUB experiment.
IMAGENET_NUM_BATCHES = 80
IMAGENET_LOADER_SEED = 2026

NUM_WORKERS = 2

# Optimization settings
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

BASE_LR = 0.001
HEAD_ONLY_LR = 0.005
FEATURE_EXTRACTION_LR = 0.005

TEMPERATURE = 2.0
LAMBDA_OLD = 1.0

WARMUP_MAX_EPOCHS = 1
FINETUNING_MAX_EPOCHS = 5
FEATURE_EXTRACTION_MAX_EPOCHS = 5
LWF_MAX_EPOCHS = 5

EARLY_STOPPING_PATIENCE = 2 if QUICK_MODE else 3
SCHEDULER_PATIENCE = 1

SAVE_MODELS = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
print("Run seeds:", RUN_SEEDS)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: a GPU is strongly recommended for the final three-seed run.")


PyTorch: 2.11.0+cu128
Device: cuda
Run seeds: [42]
GPU: Tesla T4


# Step 1 — Mount Google Drive and Define Paths

The dataset ZIP file is extracted only when the expected image folder is missing.

The notebook does not contain any private API key or Kaggle token.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/lwf"

# MIT Indoor 67 Scenes dataset
SCENES_ZIP_PATH = f"{PROJECT_ROOT}/MIT INDOOR.zip"
SCENES_EXTRACT_ROOT = f"{PROJECT_ROOT}/MIT_INDOOR"
SCENES_IMAGES_ROOT = f"{SCENES_EXTRACT_ROOT}/indoorCVPR_09/Images"
SCENES_TRAIN_SPLIT = f"{SCENES_EXTRACT_ROOT}/TrainImages.txt"
SCENES_TEST_SPLIT = f"{SCENES_EXTRACT_ROOT}/TestImages.txt"

# Existing ImageNet validation folder
IMAGENET_ROOT = f"{PROJECT_ROOT}/imagenet_validation/imagenet_validation"

# Optional output folder
OUTPUT_ROOT = f"{PROJECT_ROOT}/scenes_reproduction_outputs"

print("Scenes image root:", SCENES_IMAGES_ROOT)
print("ImageNet validation root:", IMAGENET_ROOT)


Mounted at /content/drive
Scenes image root: /content/drive/MyDrive/lwf/MIT_INDOOR/indoorCVPR_09/Images
ImageNet validation root: /content/drive/MyDrive/lwf/imagenet_validation/imagenet_validation


In [ ]:
# Extract the MIT Indoor Scenes ZIP only when needed.

if not os.path.isdir(SCENES_IMAGES_ROOT):
    print("MIT Indoor Scenes folder was not found. Extracting the ZIP file...")

    if not os.path.isfile(SCENES_ZIP_PATH):
        raise FileNotFoundError(
            f"Scenes ZIP file was not found: {SCENES_ZIP_PATH}"
        )

    os.makedirs(SCENES_EXTRACT_ROOT, exist_ok=True)

    with zipfile.ZipFile(SCENES_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(SCENES_EXTRACT_ROOT)

    print("Extraction completed.")
else:
    print("MIT Indoor Scenes is already extracted. Skipping extraction.")

required_paths = [
    SCENES_IMAGES_ROOT,
    SCENES_TRAIN_SPLIT,
    SCENES_TEST_SPLIT,
    IMAGENET_ROOT
]

for required_path in required_paths:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f"Missing required path: {required_path}")

print("All required paths were found.")


MIT Indoor Scenes is already extracted. Skipping extraction.
All required paths were found.


# Step 2 — Prepare the MIT Indoor Scenes Dataset

The official dataset contains:

- 5,360 training images
- 1,340 official test images
- 67 classes

The official training images are split internally into training and validation subsets.  
The official test set remains untouched until the final evaluation.


In [ ]:
def read_split_file(split_path):
    with open(split_path, "r", encoding="utf-8") as file:
        return [
            line.strip().replace("\\", "/")
            for line in file
            if line.strip()
        ]


def extract_class_name(relative_path):
    return relative_path.split("/")[0]


scene_classes = sorted(
    folder_name
    for folder_name in os.listdir(SCENES_IMAGES_ROOT)
    if os.path.isdir(os.path.join(SCENES_IMAGES_ROOT, folder_name))
)

class_to_idx = {
    class_name: class_index
    for class_index, class_name in enumerate(scene_classes)
}

NUM_SCENES_CLASSES = len(scene_classes)

official_train_paths = read_split_file(SCENES_TRAIN_SPLIT)
official_test_paths = read_split_file(SCENES_TEST_SPLIT)

print("Number of scene classes:", NUM_SCENES_CLASSES)
print("Official training images:", len(official_train_paths))
print("Official testing images:", len(official_test_paths))
print("Total images:", len(official_train_paths) + len(official_test_paths))

if NUM_SCENES_CLASSES != 67:
    raise ValueError(
        f"Expected 67 MIT Indoor classes, but found {NUM_SCENES_CLASSES}."
    )

if len(official_train_paths) != 5360:
    raise ValueError(
        f"Expected 5,360 official training images, but found {len(official_train_paths)}."
    )

if len(official_test_paths) != 1340:
    raise ValueError(
        f"Expected 1,340 official testing images, but found {len(official_test_paths)}."
    )


Number of scene classes: 67
Official training images: 5360
Official testing images: 1340
Total images: 6700


In [ ]:
def stratified_train_val_split(relative_paths, val_fraction, seed):
    grouped_paths = defaultdict(list)

    for relative_path in relative_paths:
        class_name = extract_class_name(relative_path)
        grouped_paths[class_name].append(relative_path)

    rng = random.Random(seed)

    train_paths = []
    val_paths = []

    for class_name in sorted(grouped_paths):
        class_paths = grouped_paths[class_name].copy()
        rng.shuffle(class_paths)

        num_val = max(1, round(len(class_paths) * val_fraction))

        val_paths.extend(class_paths[:num_val])
        train_paths.extend(class_paths[num_val:])

    rng.shuffle(train_paths)
    rng.shuffle(val_paths)

    return train_paths, val_paths


train_paths, val_paths = stratified_train_val_split(
    relative_paths=official_train_paths,
    val_fraction=VAL_FRACTION,
    seed=VAL_SPLIT_SEED
)

print("Internal training images:", len(train_paths))
print("Internal validation images:", len(val_paths))
print("Official test images:", len(official_test_paths))
print("No official test image is used for model selection.")


Internal training images: 4556
Internal validation images: 804
Official test images: 1340
No official test image is used for model selection.


In [ ]:
# Training augmentation reduces overfitting.
# Validation and testing use deterministic center crops.

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

scenes_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

scenes_eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


class MITIndoorScenesDataset(Dataset):
    def __init__(self, relative_image_paths, images_root, class_to_idx, transform=None):
        self.relative_image_paths = list(relative_image_paths)
        self.images_root = images_root
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.relative_image_paths)

    def __getitem__(self, index):
        relative_path = self.relative_image_paths[index]
        image_path = os.path.join(self.images_root, relative_path)

        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"Image file was not found: {image_path}")

        class_name = extract_class_name(relative_path)
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = MITIndoorScenesDataset(
    relative_image_paths=train_paths,
    images_root=SCENES_IMAGES_ROOT,
    class_to_idx=class_to_idx,
    transform=scenes_train_transform
)

val_dataset = MITIndoorScenesDataset(
    relative_image_paths=val_paths,
    images_root=SCENES_IMAGES_ROOT,
    class_to_idx=class_to_idx,
    transform=scenes_eval_transform
)

test_dataset = MITIndoorScenesDataset(
    relative_image_paths=official_test_paths,
    images_root=SCENES_IMAGES_ROOT,
    class_to_idx=class_to_idx,
    transform=scenes_eval_transform
)

sample_image, sample_label = train_dataset[0]

print("Sample tensor shape:", sample_image.shape)
print("Sample label:", sample_label)
print("Sample class:", scene_classes[sample_label])


Sample tensor shape: torch.Size([3, 224, 224])
Sample label: 18
Sample class: concert_hall


# Step 3 — Prepare the ImageNet Validation Dataset

ImageNet accuracy is evaluated on the first **80 validation batches** with `shuffle=False`, consistently with the protocol used in the CUB experiment.

The complete ImageNet validation dataset is loaded, while the evaluation functions stop after 80 batches only for ImageNet measurements.


In [ ]:
imagenet_eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

imagenet_full_dataset = datasets.ImageFolder(
    root=IMAGENET_ROOT,
    transform=imagenet_eval_transform
)

print("ImageNet validation images:", len(imagenet_full_dataset))
print("ImageNet classes:", len(imagenet_full_dataset.classes))
print("ImageNet evaluation batches:", IMAGENET_NUM_BATCHES)


ImageNet validation images: 50000
ImageNet classes: 1000
ImageNet evaluation batches: 80


# Step 4 — Reproducibility and DataLoader Helpers

A fresh deterministic loader is created for each training stage.  
This improves fairness between the compared methods.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic settings improve repeatability.
    # They may slightly reduce speed.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=generator
    )


def make_scenes_loaders(seed):
    train_loader = make_loader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE_SCENES,
        shuffle=True,
        seed=seed
    )

    val_loader = make_loader(
        dataset=val_dataset,
        batch_size=BATCH_SIZE_SCENES,
        shuffle=False,
        seed=seed
    )

    test_loader = make_loader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE_SCENES,
        shuffle=False,
        seed=seed
    )

    return train_loader, val_loader, test_loader


imagenet_loader = make_loader(
    dataset=imagenet_full_dataset,
    batch_size=BATCH_SIZE_IMAGENET,
    shuffle=False,
    seed=IMAGENET_LOADER_SEED
)


# Step 5 — Model Definitions

The models follow a consistent AlexNet-based design:

- `old_head`: ImageNet classifier with 1,000 outputs
- `new_head`: Scenes classifier with 67 outputs
- Shared AlexNet layers: all layers before the final ImageNet classifier

For Feature Extraction, the shared representation is frozen, dropout is disabled, and a two-layer task-specific head is trained.


In [ ]:
def initialize_linear_layers(module):
    for layer in module.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)

            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


def set_requires_grad(module, requires_grad):
    for parameter in module.parameters():
        parameter.requires_grad = requires_grad


def create_pretrained_alexnet():
    return models.alexnet(
        weights=models.AlexNet_Weights.IMAGENET1K_V1
    )


class DualHeadAlexNet(nn.Module):
    def __init__(self, pretrained_model, num_new_classes):
        super().__init__()

        self.features = copy.deepcopy(pretrained_model.features)
        self.avgpool = copy.deepcopy(pretrained_model.avgpool)

        # AlexNet layers before the original final classifier.
        self.shared_classifier = copy.deepcopy(
            nn.Sequential(*list(pretrained_model.classifier.children())[:-1])
        )

        # Original ImageNet task-specific head.
        self.old_head = copy.deepcopy(pretrained_model.classifier[6])

        # New MIT Indoor Scenes task-specific head.
        self.new_head = nn.Linear(4096, num_new_classes)
        initialize_linear_layers(self.new_head)

    def encode(self, images):
        x = self.features(images)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.shared_classifier(x)
        return x

    def forward(self, images, task):
        x = self.encode(images)

        if task == "imagenet":
            return self.old_head(x)

        if task == "scenes":
            return self.new_head(x)

        raise ValueError("task must be either 'imagenet' or 'scenes'")

    def forward_both(self, images):
        x = self.encode(images)
        return self.old_head(x), self.new_head(x)


class FeatureExtractionAlexNet(nn.Module):
    def __init__(self, pretrained_model, num_new_classes):
        super().__init__()

        self.features = copy.deepcopy(pretrained_model.features)
        self.avgpool = copy.deepcopy(pretrained_model.avgpool)

        self.shared_classifier = copy.deepcopy(
            nn.Sequential(*list(pretrained_model.classifier.children())[:-1])
        )

        # Dropout is disabled inside the frozen feature extractor.
        for layer in self.shared_classifier.modules():
            if isinstance(layer, nn.Dropout):
                layer.p = 0.0

        self.old_head = copy.deepcopy(pretrained_model.classifier[6])

        # Paper-inspired two-layer task-specific head:
        # 4096 -> 4096 -> 67
        self.new_head = nn.Sequential(
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_new_classes)
        )

        initialize_linear_layers(self.new_head)

        set_requires_grad(self.features, False)
        set_requires_grad(self.avgpool, False)
        set_requires_grad(self.shared_classifier, False)
        set_requires_grad(self.old_head, False)
        set_requires_grad(self.new_head, True)

    def encode(self, images):
        x = self.features(images)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.shared_classifier(x)
        return x

    def forward(self, images, task):
        x = self.encode(images)

        if task == "imagenet":
            return self.old_head(x)

        if task == "scenes":
            return self.new_head(x)

        raise ValueError("task must be either 'imagenet' or 'scenes'")


# Step 6 — Evaluation and Training Helpers

The best checkpoint is selected using only the internal validation set.  
The official test set is evaluated only after training is complete.


In [ ]:
classification_loss = nn.CrossEntropyLoss()


def trainable_parameters(model):
    return [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]


def current_learning_rate(optimizer):
    return optimizer.param_groups[0]["lr"]


def evaluate_standard_top1(model, dataloader, device, max_batches=None):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            predictions = logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        raise RuntimeError("No images were evaluated.")

    return 100.0 * correct / total


def evaluate_task_top1(model, dataloader, device, task, max_batches=None):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images, task=task)
            predictions = logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        raise RuntimeError("No images were evaluated.")

    return 100.0 * correct / total


def train_supervised_epoch(model, dataloader, optimizer, device, task="scenes"):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images, task=task)
        loss = classification_loss(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit_supervised_with_validation(
    model,
    train_loader,
    val_loader,
    device,
    max_epochs,
    learning_rate,
    weight_decay,
    label
):
    optimizer = optim.SGD(
        trainable_parameters(model),
        lr=learning_rate,
        momentum=MOMENTUM,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.1,
        patience=SCHEDULER_PATIENCE
    )

    best_val_accuracy = float("-inf")
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, train_accuracy = train_supervised_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            device=device,
            task="scenes"
        )

        val_accuracy = evaluate_task_top1(
            model=model,
            dataloader=val_loader,
            device=device,
            task="scenes"
        )

        learning_rate_now = current_learning_rate(optimizer)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_accuracy,
            "learning_rate": learning_rate_now
        })

        print(
            f"{label} | Epoch [{epoch}/{max_epochs}] | "
            f"LR: {learning_rate_now:.6f} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%"
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        scheduler.step(val_accuracy)

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"{label} | Early stopping triggered.")
            break

    if best_state is None:
        raise RuntimeError(f"{label}: no checkpoint was saved.")

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_accuracy


# Step 7 — LwF Loss and Training Helpers

LwF minimizes:

- the Scenes classification loss
- the ImageNet response-preservation loss

The teacher remains frozen.  
After the warm-up stage, the student shared layers, old head, and new head are jointly optimized.


In [ ]:
def distillation_loss(student_old_logits, teacher_old_logits, temperature):
    teacher_probabilities = F.softmax(
        teacher_old_logits / temperature,
        dim=1
    )

    student_log_probabilities = F.log_softmax(
        student_old_logits / temperature,
        dim=1
    )

    return F.kl_div(
        student_log_probabilities,
        teacher_probabilities,
        reduction="batchmean"
    ) * (temperature ** 2)


def train_lwf_epoch(student_model, teacher_model, dataloader, optimizer, device):
    student_model.train()
    teacher_model.eval()

    running_total_loss = 0.0
    running_new_loss = 0.0
    running_old_loss = 0.0

    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.no_grad():
            teacher_old_logits = teacher_model(images)

        student_old_logits, student_new_logits = student_model.forward_both(images)

        loss_new = classification_loss(
            student_new_logits,
            labels
        )

        loss_old = distillation_loss(
            student_old_logits=student_old_logits,
            teacher_old_logits=teacher_old_logits,
            temperature=TEMPERATURE
        )

        total_loss = loss_new + LAMBDA_OLD * loss_old

        total_loss.backward()
        optimizer.step()

        batch_size = images.size(0)

        running_total_loss += total_loss.item() * batch_size
        running_new_loss += loss_new.item() * batch_size
        running_old_loss += loss_old.item() * batch_size

        predictions = student_new_logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return {
        "total_loss": running_total_loss / total,
        "new_loss": running_new_loss / total,
        "old_loss": running_old_loss / total,
        "train_accuracy": 100.0 * correct / total
    }


def fit_lwf_with_validation(
    student_model,
    teacher_model,
    train_loader,
    val_loader,
    device,
    max_epochs,
    learning_rate,
    label
):
    # Jointly optimize θs, θo, and θn after warm-up.
    set_requires_grad(student_model, True)

    optimizer = optim.SGD(
        trainable_parameters(student_model),
        lr=learning_rate,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.1,
        patience=SCHEDULER_PATIENCE
    )

    best_val_accuracy = float("-inf")
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_metrics = train_lwf_epoch(
            student_model=student_model,
            teacher_model=teacher_model,
            dataloader=train_loader,
            optimizer=optimizer,
            device=device
        )

        val_accuracy = evaluate_task_top1(
            model=student_model,
            dataloader=val_loader,
            device=device,
            task="scenes"
        )

        learning_rate_now = current_learning_rate(optimizer)

        history.append({
            "epoch": epoch,
            "total_loss": train_metrics["total_loss"],
            "new_loss": train_metrics["new_loss"],
            "old_loss": train_metrics["old_loss"],
            "train_accuracy": train_metrics["train_accuracy"],
            "val_accuracy": val_accuracy,
            "learning_rate": learning_rate_now
        })

        print(
            f"{label} | Epoch [{epoch}/{max_epochs}] | "
            f"LR: {learning_rate_now:.6f} | "
            f"Total Loss: {train_metrics['total_loss']:.4f} | "
            f"New Loss: {train_metrics['new_loss']:.4f} | "
            f"Old Loss: {train_metrics['old_loss']:.4f} | "
            f"Train Acc: {train_metrics['train_accuracy']:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%"
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(student_model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        scheduler.step(val_accuracy)

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"{label} | Early stopping triggered.")
            break

    if best_state is None:
        raise RuntimeError(f"{label}: no checkpoint was saved.")

    student_model.load_state_dict(best_state)

    return student_model, pd.DataFrame(history), best_val_accuracy


# Step 8 — Measure the ImageNet Baseline

The pretrained AlexNet model is evaluated once on the first **80 ImageNet validation batches**.


In [ ]:
seed_everything(IMAGENET_LOADER_SEED)

baseline_model = create_pretrained_alexnet().to(device)
baseline_model.eval()

baseline_imagenet_accuracy = evaluate_standard_top1(
    model=baseline_model,
    dataloader=imagenet_loader,
    device=device,
    max_batches=IMAGENET_NUM_BATCHES
)

print(
    f"Baseline ImageNet Accuracy on the first "
    f"{IMAGENET_NUM_BATCHES} batches: "
    f"{baseline_imagenet_accuracy:.2f}%"
)

del baseline_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:02<00:00, 111MB/s]


Baseline ImageNet Accuracy on the first 80 batches: 62.89%


# Step 9 — Run Fine-Tuning, Feature Extraction, and LwF

For each seed:

1. A new-task head is warmed up while the pretrained representation is frozen.
2. Fine-Tuning starts from the warmed-up state and updates the shared representation and the Scenes head while keeping the old ImageNet head fixed.
3. Feature Extraction is trained separately using frozen ImageNet features and a two-layer Scenes head.
4. LwF starts from the same warmed-up state and jointly optimizes shared parameters, the old head, and the new head using classification and distillation losses.
5. The best validation checkpoint of each method is evaluated once on the official Scenes test set and on the fixed ImageNet subset.


In [ ]:
def append_result(
    result_rows,
    seed,
    method,
    old_task_accuracy,
    new_task_accuracy,
    best_val_accuracy
):
    result_rows.append({
        "Seed": seed,
        "Method": method,
        "ImageNet Old Task Accuracy (%)": old_task_accuracy,
        "Scenes Official Test Accuracy (%)": new_task_accuracy,
        "Best Scenes Validation Accuracy (%)": best_val_accuracy,
        "ImageNet Difference from Baseline (%)": (
            old_task_accuracy - baseline_imagenet_accuracy
        )
    })


all_results = []
all_histories = {}
saved_model_states = {}

for run_seed in RUN_SEEDS:
    print()
    print("=" * 90)
    print(f"STARTING RUN WITH SEED {run_seed}")
    print("=" * 90)

    seed_everything(run_seed)

    # Fresh pretrained model for this run.
    pretrained_model = create_pretrained_alexnet().to(device)
    pretrained_model.eval()

    # -------------------------------------------------
    # Stage A: shared warm-up for Fine-Tuning and LwF
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE A — Warm-up: train only the new Scenes head")
    print("-" * 90)

    warmup_model = DualHeadAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_SCENES_CLASSES
    ).to(device)

    set_requires_grad(warmup_model, False)
    set_requires_grad(warmup_model.new_head, True)

    warmup_train_loader, warmup_val_loader, _ = make_scenes_loaders(
        seed=run_seed
    )

    warmup_model, warmup_history, warmup_best_val = fit_supervised_with_validation(
        model=warmup_model,
        train_loader=warmup_train_loader,
        val_loader=warmup_val_loader,
        device=device,
        max_epochs=WARMUP_MAX_EPOCHS,
        learning_rate=HEAD_ONLY_LR,
        weight_decay=WEIGHT_DECAY,
        label=f"Seed {run_seed} | Warm-up"
    )

    warmup_state = copy.deepcopy(warmup_model.state_dict())
    all_histories[(run_seed, "Warm-up")] = warmup_history

    del warmup_train_loader, warmup_val_loader

    # -------------------------------------------------
    # Stage B: Fine-Tuning
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE B — Fine-Tuning")
    print("-" * 90)

    ft_model = DualHeadAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_SCENES_CLASSES
    ).to(device)

    ft_model.load_state_dict(warmup_state)

    # Fine-Tuning updates shared parameters and the new head.
    # The old ImageNet head is retained for evaluation but remains fixed.
    set_requires_grad(ft_model, True)
    set_requires_grad(ft_model.old_head, False)

    ft_train_loader, ft_val_loader, ft_test_loader = make_scenes_loaders(
        seed=run_seed + 100
    )

    ft_model, ft_history, ft_best_val = fit_supervised_with_validation(
        model=ft_model,
        train_loader=ft_train_loader,
        val_loader=ft_val_loader,
        device=device,
        max_epochs=FINETUNING_MAX_EPOCHS,
        learning_rate=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        label=f"Seed {run_seed} | Fine-Tuning"
    )

    ft_scenes_test_accuracy = evaluate_task_top1(
        model=ft_model,
        dataloader=ft_test_loader,
        device=device,
        task="scenes"
    )

    ft_imagenet_accuracy = evaluate_task_top1(
        model=ft_model,
        dataloader=imagenet_loader,
        device=device,
        task="imagenet",
        max_batches=IMAGENET_NUM_BATCHES
    )

    append_result(
        result_rows=all_results,
        seed=run_seed,
        method="Fine-Tuning",
        old_task_accuracy=ft_imagenet_accuracy,
        new_task_accuracy=ft_scenes_test_accuracy,
        best_val_accuracy=ft_best_val
    )

    all_histories[(run_seed, "Fine-Tuning")] = ft_history

    if SAVE_MODELS:
        saved_model_states[(run_seed, "Fine-Tuning")] = copy.deepcopy(
            ft_model.state_dict()
        )

    print(
        f"Seed {run_seed} | Fine-Tuning | "
        f"ImageNet: {ft_imagenet_accuracy:.2f}% | "
        f"Scenes official test: {ft_scenes_test_accuracy:.2f}%"
    )

    del ft_model, ft_train_loader, ft_val_loader, ft_test_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -------------------------------------------------
    # Stage C: Feature Extraction
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE C — Feature Extraction")
    print("-" * 90)

    seed_everything(run_seed)

    feature_model = FeatureExtractionAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_SCENES_CLASSES
    ).to(device)

    fe_train_loader, fe_val_loader, fe_test_loader = make_scenes_loaders(
        seed=run_seed + 100
    )

    feature_model, fe_history, fe_best_val = fit_supervised_with_validation(
        model=feature_model,
        train_loader=fe_train_loader,
        val_loader=fe_val_loader,
        device=device,
        max_epochs=FEATURE_EXTRACTION_MAX_EPOCHS,
        learning_rate=FEATURE_EXTRACTION_LR,
        weight_decay=WEIGHT_DECAY,
        label=f"Seed {run_seed} | Feature Extraction"
    )

    fe_scenes_test_accuracy = evaluate_task_top1(
        model=feature_model,
        dataloader=fe_test_loader,
        device=device,
        task="scenes"
    )

    # Explicitly measure the preserved ImageNet accuracy.
    fe_imagenet_accuracy = evaluate_task_top1(
        model=feature_model,
        dataloader=imagenet_loader,
        device=device,
        task="imagenet",
        max_batches=IMAGENET_NUM_BATCHES
    )

    append_result(
        result_rows=all_results,
        seed=run_seed,
        method="Feature Extraction",
        old_task_accuracy=fe_imagenet_accuracy,
        new_task_accuracy=fe_scenes_test_accuracy,
        best_val_accuracy=fe_best_val
    )

    all_histories[(run_seed, "Feature Extraction")] = fe_history

    if SAVE_MODELS:
        saved_model_states[(run_seed, "Feature Extraction")] = copy.deepcopy(
            feature_model.state_dict()
        )

    print(
        f"Seed {run_seed} | Feature Extraction | "
        f"ImageNet: {fe_imagenet_accuracy:.2f}% | "
        f"Scenes official test: {fe_scenes_test_accuracy:.2f}%"
    )

    del feature_model, fe_train_loader, fe_val_loader, fe_test_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -------------------------------------------------
    # Stage D: Learning without Forgetting
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE D — Learning without Forgetting")
    print("-" * 90)

    student_model = DualHeadAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_SCENES_CLASSES
    ).to(device)

    student_model.load_state_dict(warmup_state)

    teacher_model = create_pretrained_alexnet().to(device)
    teacher_model.eval()
    set_requires_grad(teacher_model, False)

    lwf_train_loader, lwf_val_loader, lwf_test_loader = make_scenes_loaders(
        seed=run_seed + 100
    )

    student_model, lwf_history, lwf_best_val = fit_lwf_with_validation(
        student_model=student_model,
        teacher_model=teacher_model,
        train_loader=lwf_train_loader,
        val_loader=lwf_val_loader,
        device=device,
        max_epochs=LWF_MAX_EPOCHS,
        learning_rate=BASE_LR,
        label=f"Seed {run_seed} | LwF"
    )

    lwf_scenes_test_accuracy = evaluate_task_top1(
        model=student_model,
        dataloader=lwf_test_loader,
        device=device,
        task="scenes"
    )

    lwf_imagenet_accuracy = evaluate_task_top1(
        model=student_model,
        dataloader=imagenet_loader,
        device=device,
        task="imagenet",
        max_batches=IMAGENET_NUM_BATCHES
    )

    append_result(
        result_rows=all_results,
        seed=run_seed,
        method="LwF",
        old_task_accuracy=lwf_imagenet_accuracy,
        new_task_accuracy=lwf_scenes_test_accuracy,
        best_val_accuracy=lwf_best_val
    )

    all_histories[(run_seed, "LwF")] = lwf_history

    if SAVE_MODELS:
        saved_model_states[(run_seed, "LwF")] = copy.deepcopy(
            student_model.state_dict()
        )

    print(
        f"Seed {run_seed} | LwF | "
        f"ImageNet: {lwf_imagenet_accuracy:.2f}% | "
        f"Scenes official test: {lwf_scenes_test_accuracy:.2f}%"
    )

    del (
        student_model,
        teacher_model,
        pretrained_model,
        warmup_model,
        lwf_train_loader,
        lwf_val_loader,
        lwf_test_loader
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print("All runs completed successfully.")



STARTING RUN WITH SEED 42

------------------------------------------------------------------------------------------
STAGE A — Warm-up: train only the new Scenes head
------------------------------------------------------------------------------------------
Seed 42 | Warm-up | Epoch [1/1] | LR: 0.005000 | Train Loss: 3.4345 | Train Acc: 34.26% | Val Acc: 46.02%

------------------------------------------------------------------------------------------
STAGE B — Fine-Tuning
------------------------------------------------------------------------------------------
Seed 42 | Fine-Tuning | Epoch [1/5] | LR: 0.001000 | Train Loss: 1.6589 | Train Acc: 53.95% | Val Acc: 49.00%
Seed 42 | Fine-Tuning | Epoch [2/5] | LR: 0.001000 | Train Loss: 1.2222 | Train Acc: 64.62% | Val Acc: 53.23%
Seed 42 | Fine-Tuning | Epoch [3/5] | LR: 0.001000 | Train Loss: 1.0131 | Train Acc: 70.83% | Val Acc: 54.10%
Seed 42 | Fine-Tuning | Epoch [4/5] | LR: 0.001000 | Train Loss: 0.8315 | Train Acc: 75.22% | Val A

# Step 10 — Results Tables

The first table contains the result of every seed.  
The second table reports the final mean and sample standard deviation.


In [ ]:
results_df = pd.DataFrame(all_results)

method_order = [
    "Fine-Tuning",
    "Feature Extraction",
    "LwF"
]

results_df["Method"] = pd.Categorical(
    results_df["Method"],
    categories=method_order,
    ordered=True
)

results_df = results_df.sort_values(
    by=["Method", "Seed"]
).reset_index(drop=True)

print("Per-seed results")
display(results_df.round(2))


Per-seed results


,Seed,Method,ImageNet Old Task Accuracy (%),Scenes Official Test Accuracy (%),Best Scenes Validation Accuracy (%),ImageNet Difference from Baseline (%)
0,42,Fine-Tuning,48.69,50.82,54.10,-14.20
1,42,Feature Extraction,62.89,55.00,57.21,0.00
2,42,LwF,59.82,56.27,56.97,-3.07


In [ ]:
def sample_std(series):
    if len(series) <= 1:
        return 0.0

    return series.std(ddof=1)


summary_df = (
    results_df
    .groupby("Method", observed=True)
    .agg(
        ImageNet_Old_Mean=("ImageNet Old Task Accuracy (%)", "mean"),
        ImageNet_Old_Std=("ImageNet Old Task Accuracy (%)", sample_std),
        Scenes_Test_Mean=("Scenes Official Test Accuracy (%)", "mean"),
        Scenes_Test_Std=("Scenes Official Test Accuracy (%)", sample_std),
        ImageNet_Difference_Mean=("ImageNet Difference from Baseline (%)", "mean"),
        ImageNet_Difference_Std=("ImageNet Difference from Baseline (%)", sample_std),
        Best_Validation_Mean=("Best Scenes Validation Accuracy (%)", "mean"),
        Best_Validation_Std=("Best Scenes Validation Accuracy (%)", sample_std)
    )
    .reset_index()
)

baseline_row = pd.DataFrame({
    "Method": ["Baseline ImageNet"],
    "ImageNet_Old_Mean": [baseline_imagenet_accuracy],
    "ImageNet_Old_Std": [0.0],
    "Scenes_Test_Mean": [np.nan],
    "Scenes_Test_Std": [np.nan],
    "ImageNet_Difference_Mean": [0.0],
    "ImageNet_Difference_Std": [0.0],
    "Best_Validation_Mean": [np.nan],
    "Best_Validation_Std": [np.nan]
})

summary_with_baseline_df = pd.concat(
    [baseline_row, summary_df],
    ignore_index=True
)

print("Final summary: mean ± sample standard deviation")
display(summary_with_baseline_df.round(2))


Final summary: mean ± sample standard deviation


,Method,ImageNet_Old_Mean,ImageNet_Old_Std,Scenes_Test_Mean,Scenes_Test_Std,ImageNet_Difference_Mean,ImageNet_Difference_Std,Best_Validation_Mean,Best_Validation_Std
0,Baseline ImageNet,62.89,0.0,NaN,NaN,0.00,0.0,NaN,NaN
1,Fine-Tuning,48.69,0.0,50.82,0.0,-14.20,0.0,54.10,0.0
2,Feature Extraction,62.89,0.0,55.00,0.0,0.00,0.0,57.21,0.0
3,LwF,59.82,0.0,56.27,0.0,-3.07,0.0,56.97,0.0


In [ ]:
def format_mean_std(mean_value, std_value):
    if pd.isna(mean_value):
        return "—"

    return f"{mean_value:.2f} ± {std_value:.2f}"


report_table = pd.DataFrame({
    "Method": summary_with_baseline_df["Method"],
    "ImageNet Old Task Accuracy (%)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            summary_with_baseline_df["ImageNet_Old_Mean"],
            summary_with_baseline_df["ImageNet_Old_Std"]
        )
    ],
    "Scenes Official Test Accuracy (%)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            summary_with_baseline_df["Scenes_Test_Mean"],
            summary_with_baseline_df["Scenes_Test_Std"]
        )
    ],
    "ImageNet Difference from Baseline (%)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            summary_with_baseline_df["ImageNet_Difference_Mean"],
            summary_with_baseline_df["ImageNet_Difference_Std"]
        )
    ]
})

print("Copy-friendly report table")
display(report_table)


Copy-friendly report table


,Method,ImageNet Old Task Accuracy (%),Scenes Official Test Accuracy (%),ImageNet Difference from Baseline (%)
0,Baseline ImageNet,62.89 ± 0.00,—,0.00 ± 0.00
1,Fine-Tuning,48.69 ± 0.00,50.82 ± 0.00,-14.20 ± 0.00
2,Feature Extraction,62.89 ± 0.00,55.00 ± 0.00,0.00 ± 0.00
3,LwF,59.82 ± 0.00,56.27 ± 0.00,-3.07 ± 0.00


# Step 11 — Save Results and Optional Model Weights

The CSV files are useful for the report and for GitHub submission.

Model saving is disabled by default because AlexNet checkpoints are large.  
To save them, set `SAVE_MODELS = True` in Step 0 before running the notebook.


In [ ]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)

per_seed_csv_path = f"{OUTPUT_ROOT}/scenes_per_seed_results.csv"
summary_csv_path = f"{OUTPUT_ROOT}/scenes_summary_results.csv"
report_csv_path = f"{OUTPUT_ROOT}/scenes_report_table.csv"

results_df.to_csv(per_seed_csv_path, index=False)
summary_with_baseline_df.to_csv(summary_csv_path, index=False)
report_table.to_csv(report_csv_path, index=False)

print("Saved:", per_seed_csv_path)
print("Saved:", summary_csv_path)
print("Saved:", report_csv_path)

if SAVE_MODELS:
    model_output_root = f"{OUTPUT_ROOT}/saved_models"
    os.makedirs(model_output_root, exist_ok=True)

    for (seed, method), state_dict in saved_model_states.items():
        safe_method_name = method.lower().replace(" ", "_").replace("-", "_")
        save_path = f"{model_output_root}/{safe_method_name}_seed_{seed}.pth"

        torch.save(state_dict, save_path)
        print("Saved:", save_path)


Saved: /content/drive/MyDrive/lwf/scenes_reproduction_outputs/scenes_per_seed_results.csv
Saved: /content/drive/MyDrive/lwf/scenes_reproduction_outputs/scenes_summary_results.csv
Saved: /content/drive/MyDrive/lwf/scenes_reproduction_outputs/scenes_report_table.csv


# Step 12 — Optional Diagnostic: Verify Feature Extraction Preservation

Because Feature Extraction freezes the original representation and old head, its ImageNet predictions should match the pretrained baseline exactly.

This check compares the predictions sample-by-sample on the fixed ImageNet subset.  
Run it only when you want an additional diagnostic.


In [ ]:
def compare_old_task_predictions(
    model_a,
    model_b,
    dataloader,
    device,
    max_batches=IMAGENET_NUM_BATCHES
):
    model_a.eval()
    model_b.eval()

    total = 0
    same_predictions = 0

    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device, non_blocking=True)

            logits_a = model_a(images)
            logits_b = model_b(images, task="imagenet")

            predictions_a = logits_a.argmax(dim=1)
            predictions_b = logits_b.argmax(dim=1)

            same_predictions += (
                predictions_a == predictions_b
            ).sum().item()

            total += images.size(0)

    if total == 0:
        raise RuntimeError("No images were evaluated.")

    print("Compared images:", total)
    print("Identical predictions:", same_predictions)
    print("Agreement:", 100.0 * same_predictions / total, "%")


# Example usage after creating a feature-extraction model:
#
# diagnostic_baseline = create_pretrained_alexnet().to(device)
# diagnostic_feature_model = FeatureExtractionAlexNet(
#     pretrained_model=diagnostic_baseline,
#     num_new_classes=NUM_SCENES_CLASSES
# ).to(device)
#
# compare_old_task_predictions(
#     model_a=diagnostic_baseline,
#     model_b=diagnostic_feature_model,
#     dataloader=imagenet_loader,
#     device=device,
#     max_batches=IMAGENET_NUM_BATCHES
# )
